# 第 4 章 · RoPE 旋转位置编码

> 📄 配套动画网页：`../04_rope.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第二部分 · 模型

注意力本身不知道词的顺序。RoPE 用「旋转」这个几何操作，把位置信息优雅地注入到 Q 和 K 里。

---


## 🎯 本章目标

- 理解 RoPE 的几何直觉：位置 = 旋转角，内积只依赖相对距离
- 掌握 `torch.outer`、`torch.cos/sin`、切片翻转 `rotate_half`
- 掌握 `unsqueeze` 的广播维度对齐技巧
- 能从零实现 `precompute_freqs_cis` 与 `apply_rotary_pos_emb`

---


## 📖 背景：位置信息怎么塞进注意力

把向量的每两个维度看成一个复数平面上的点。位置 `m` 的 token，第 `i` 组维度旋转 `m·θᵢ`：

$$\theta_i = \frac{1}{10000^{2i/d}}, \qquad
\begin{bmatrix} x' \\ y' \end{bmatrix} =
\begin{bmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{bmatrix}
\begin{bmatrix} x \\ y \end{bmatrix}$$

因为旋转是等距变换，两个向量都旋转后，它们的内积只依赖角度差 `(m−n)·θ`——**也就是相对位置**。

---



## 🖼 官方配图

![不同 RoPE 配置下的困惑度对比——YaRN 外推能让模型在更长上下文上保持稳定](../images/rope_ppl.png)

*不同 RoPE 配置下的困惑度对比——YaRN 外推能让模型在更长上下文上保持稳定*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 4.1 `torch.arange` 与 `torch.outer`

`outer(t, f)` 会算出所有位置 × 所有频率的组合矩阵。

In [ ]:
import torch, math

t = torch.arange(6).float()              # 位置 0..5
f = 1.0 / (10000 ** (torch.arange(0, 8, 2).float() / 8))   # 4 个频率
print("位置 t :", t)
print("频率 f :", f)

freqs = torch.outer(t, f)                # [6, 4]
print("freqs 形状:", freqs.shape)
print(freqs[:2])

### 4.2 `torch.cos` / `torch.sin` 与 `torch.cat`

RoPE 把 cos/sin 各复制一份再拼接，好让它们能直接和 head_dim 维度对齐。

In [ ]:
freqs = torch.randn(6, 4)
cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
print("cos 形状:", cos.shape, " (4 -> 8，因为拼了两份)")

### 4.3 `rotate_half`：切片翻转

RoPE 的旋转可以写成 `x·cos + rotate_half(x)·sin`。`rotate_half` 把后半段取负放到前面。

In [ ]:
def rotate_half(x):
    # x 最后一维是 d，切成两半
    return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)

x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
print("原始      :", x.tolist())
print("rotate_half:", rotate_half(x).tolist(), "  <- 期望 [-3, -4, 1, 2]")

### 4.4 `unsqueeze` 广播对齐

`cos` 的形状是 `[seq, head_dim]`，而 `q` 是 `[batch, seq, heads, head_dim]`。
要在 `seq` 与 `heads` 之间插一个维度才能广播。

In [ ]:
q = torch.randn(2, 6, 8, 16)      # [B, S, H, D]
cos = torch.randn(6, 16)          # [S, D]

# 直接在 dim=1 插入 -> [S, 1, D]，可以和 [B, S, H, D] 广播
print("unsqueeze(1):", cos.unsqueeze(1).shape)

# 验证：乘法能跑通
out = q * cos.unsqueeze(1)
print("广播相乘结果:", out.shape)

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

RoPE 有两个函数要写：预计算频率表、把旋转应用到 Q/K。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 4.5 实现 `precompute_freqs_cis`

**要求**：
1. 生成频率 `freqs = 1 / (theta ** (arange(0, dim, 2) / dim))`
2. `freqs = outer(arange(end), freqs)` 得到 `[end, dim/2]`
3. 算 `cos` 和 `sin`，各复制一份 `cat` 成 `dim` 维
4. 返回 `(freqs_cos, freqs_sin)`

（进阶：可选实现 YaRN 的 `rope_scaling` 分支，网页里有公式。）

In [ ]:
# ======== 你的实现：precompute_freqs_cis ========
#
# 【要求】
#   - 签名：precompute_freqs_cis(dim, end=32768, rope_base=1e6)
#   - freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:dim//2].float() / dim))
#   - freqs = torch.outer(torch.arange(end).float(), freqs)
#   - freqs_cos = torch.cat([cos, cos], dim=-1)；sin 同理
#   - 返回 (freqs_cos, freqs_sin)，形状都是 [end, dim]
#
# 【可用到的函数 / 方法】
#   - `torch.arange`、`torch.outer`
#   - `torch.cos`、`torch.sin`、`torch.cat`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：precompute_freqs_cis ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    cos, sin = precompute_freqs_cis(dim=8, end=16, rope_base=1e6)
    assert tuple(cos.shape) == (16, 8), f"cos 形状应为 (16, 8)，得到 {tuple(cos.shape)}"
    assert tuple(sin.shape) == (16, 8), f"sin 形状应为 (16, 8)，得到 {tuple(sin.shape)}"
    close(cos[0], torch.ones(8), tol=1e-5, msg="位置 0 的旋转角为 0，cos 应全为 1")
    close(sin[0], torch.zeros(8), tol=1e-5, msg="位置 0 的旋转角为 0，sin 应全为 0")
    close(cos[:, :4], cos[:, 4:], tol=1e-6, msg="cos 前后两半应相同（cat 了两份）")
    close(sin[:, :4], sin[:, 4:], tol=1e-6, msg="sin 前后两半应相同（cat 了两份）")
    close(cos ** 2 + sin ** 2, torch.ones(16, 8), tol=1e-4, msg="cos² + sin² 应恒为 1")

check("precompute_freqs_cis 形状与边界值", _t)

### 4.6 实现 `apply_rotary_pos_emb`

**要求**：给定 `q, k, cos, sin`，返回旋转后的 `q_embed, k_embed`。
公式：`x_embed = x * cos + rotate_half(x) * sin`。

**注意**：`cos`/`sin` 需要 `unsqueeze` 到合适的维度才能和 q/k 广播。

In [ ]:
# ======== 你的实现：apply_rotary_pos_emb ========
#
# 【要求】
#   - 内部先定义 rotate_half(x)
#   - q_embed = q * cos.unsqueeze(1) + rotate_half(q) * sin.unsqueeze(1)
#   - k_embed 同理
#   - 最后 .to(q.dtype) 保证类型一致
#
# 【可用到的函数 / 方法】
#   - 上面写过的 `rotate_half`
#   - `Tensor.unsqueeze`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：apply_rotary_pos_emb ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    cos, sin = precompute_freqs_cis(dim=8, end=8, rope_base=1e6)
    q = torch.randn(1, 8, 2, 8)
    k = torch.randn(1, 8, 2, 8)
    qe, ke = apply_rotary_pos_emb(q, k, cos, sin)
    assert tuple(qe.shape) == tuple(q.shape), f"q 形状应保持不变，得到 {tuple(qe.shape)}"
    assert tuple(ke.shape) == tuple(k.shape), f"k 形状应保持不变，得到 {tuple(ke.shape)}"
    close(qe[:, 0], q[:, 0], tol=1e-5, msg="位置 0 旋转角为 0，输出应等于输入")
    close(qe.norm(dim=-1), q.norm(dim=-1), tol=1e-4, msg="旋转是等距变换，向量范数应保持不变")
    close(ke.norm(dim=-1), k.norm(dim=-1), tol=1e-4)

check("apply_rotary_pos_emb 形状 / 位置0不变 / 保范数", _t)

### 4.7 验证相对位置不变性

这是 RoPE 最核心的性质，务必自己验证一次：
构造两个向量 q、k，分别旋转 m 和 n 位，内积应该只依赖 `(m−n)`。

In [ ]:
# ======== 你的实现：verify_rope ========
#
# 【要求】
#   - 固定 q、k 两个向量（head_dim=8）
#   - 取 (m, n) = (0, 0)、(1, 1)、(2, 2) → 内积应该相同
#   - 取 (m, n) = (0, 3)、(2, 5)、(10, 13) → 内积应该相同（相对距离都是 3）
#   - 打印各组的点积，确认同一相对距离结果一致
#
# 【可用到的函数 / 方法】
#   - 用你的 `precompute_freqs_cis` 取出对应位置的 cos/sin 切片
#   - `torch.dot` 或 `(a * b).sum()`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能画出「位置 → 旋转角」的几何图景
- [ ] `rotate_half` 的输出顺序对吗？（`[-后半, 前半]`）
- [ ] `unsqueeze(1)` 的作用说清楚了吗？
- [ ] 相对位置不变性验证通过了吗？

**思考题**：为什么不同维度要用不同的 θ，而不是所有维度共用一个？
